# 第1讲：交通大数据概述（学生操练）

        > 课程：《交通大数据分析与应用》  
        > 数据：课程模拟数据，不是实际监测数据  
        > 建议用时：15分钟

        ## 目标

        1. 在三类数据源中选择一种，确认观察对象、时间粒度和空间粒度；
2. 修改SELECTED_SOURCE，重新生成数据卡片；
3. 写出该数据源能够回答的问题和不能直接回答的问题各一项；

        代码可以直接运行；请按`TODO`修改参数、核对输出并完成解释。

## 1. Setup｜环境、路径与参数

In [ ]:
from __future__ import annotations

from pathlib import Path
import warnings

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

warnings.filterwarnings("ignore", category=FutureWarning)
DATA_DIR = Path("data")
OUTPUT_DIR = Path("outputs")
OUTPUT_DIR.mkdir(exist_ok=True)
plt.rcParams.update({"figure.dpi": 120, "axes.grid": True, "grid.alpha": 0.25})
RANDOM_STATE = 42
print("环境已就绪；数据目录：", DATA_DIR.resolve())

## 2. 读取三类交通数据

In [ ]:
detector = pd.read_csv(DATA_DIR / "traffic_15min.csv", parse_dates=["timestamp"])
probe = pd.read_csv(DATA_DIR / "probe_trips.csv", parse_dates=["start_time"])
incidents = pd.read_csv(DATA_DIR / "incident_events.csv", parse_dates=["start_time"])
catalog = pd.read_csv(DATA_DIR / "source_catalog.csv")
print({"固定断面记录": len(detector), "浮动车行程": len(probe), "事件记录": len(incidents)})
display(catalog)

## 3. 比较覆盖范围

In [ ]:
daily_coverage = (detector.assign(date=detector["timestamp"].dt.date)
                  .groupby("detector_id")["date"].nunique().sort_values())
ax = daily_coverage.plot(kind="bar", color="#2F5597", title="Observed days by detector")
ax.set_xlabel("Detector"); ax.set_ylabel("Days")
plt.tight_layout(); plt.savefig(OUTPUT_DIR / "lesson01_source_coverage.png"); plt.show()
display(daily_coverage.rename("observed_days").to_frame())

## 4. 生成数据卡片

In [ ]:
SELECTED_SOURCE = "固定断面检测器"  # TODO：改为“浮动车”或“交通事件记录”
card = catalog.loc[catalog["source"] == SELECTED_SOURCE].T
if card.empty:
    raise ValueError("SELECTED_SOURCE必须来自source_catalog.csv")
display(card)
print("可回答：", card.loc["can_answer"].iloc[0])
print("不能直接回答：", card.loc["cannot_answer"].iloc[0])

## 5. 完成自检

In [ ]:
checks = {
    "三类数据均已读取": all(len(x) > 0 for x in [detector, probe, incidents]),
    "数据卡片只有一列": card.shape[1] == 1,
    "覆盖图已生成": (OUTPUT_DIR / "lesson01_source_coverage.png").exists(),
}
status = "PASS" if all(checks.values()) else "CHECK"
pd.Series(checks).to_csv(OUTPUT_DIR / "lesson01_checks.csv", header=["passed"])
(OUTPUT_DIR / "自检结果.txt").write_text(status, encoding="utf-8")
print(status, checks)

## Checks｜当堂记录

        - 在三类数据源中选择一种，确认观察对象、时间粒度和空间粒度
- 修改SELECTED_SOURCE，重新生成数据卡片
- 写出该数据源能够回答的问题和不能直接回答的问题各一项

        **预期结果：** 一张数据覆盖图、一份数据卡片，以及两句有依据的适用性判断。

        **完成标准：** 能说清一行数据表示什么、覆盖了谁、遗漏了谁；Notebook可从头运行并生成PASS。

        请在课堂记录中写下：改了什么参数、结果发生了什么变化、这个变化在交通问题中意味着什么。